# Demo (локальный запуск): построение карусели i2i

Этот ноутбук рассчитан на запуск из Cursor/VSCode на твоей локальной машине.
Никаких Drive/zip — все артефакты уже лежат в репозитории.

**Перед запуском:** в правом верхнем углу выбери kernel `.venv` (если ноутбук открылся не с ним).

**Что делает:**
1. Поднимает inference-контекст: каталог + эмбеддинги (memmap, 400 МБ) + coview + LTR-модель (~6 сек).
2. Строит карусели K=12 для нескольких популярных статей-источников.
3. Опционально — сравнивает выдачу LTR vs fallback-эвристика на одном источнике.

## 1. Переходим в корень репо

In [1]:
import os, sys, pathlib

ROOT = pathlib.Path.cwd()
while ROOT != ROOT.parent and not (ROOT / 'serve_carousel.py').exists():
    ROOT = ROOT.parent
assert (ROOT / 'serve_carousel.py').exists(), 'не нашёл serve_carousel.py — открой ноутбук изнутри репо'

os.chdir(ROOT)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

print('root:', ROOT)
print('artifacts present:')
for p in [
    'models/ltr_lgbm.pkl',
    'coview_cache/coview_index.pkl.gz',
    'embeddings_cache/embeddings.f32',
    'tj_article.csv',
]:
    ok = (ROOT / p).exists()
    size_mb = (ROOT / p).stat().st_size / 1024 / 1024 if ok else 0
    print(f'  [{"OK" if ok else "MISS"}]  {p}  ({size_mb:.1f} MB)' if ok else f'  [MISS] {p}')

root: /Users/nkozheko/T-Ж рекомендации
artifacts present:
  [OK]  models/ltr_lgbm.pkl  (8.0 MB)
  [OK]  coview_cache/coview_index.pkl.gz  (12.7 MB)
  [OK]  embeddings_cache/embeddings.f32  (404.0 MB)
  [OK]  tj_article.csv  (122.4 MB)


## 2. Поднимаем inference-контекст

Первый запуск занимает ~6 секунд (читает каталог, эмбеддинги, coview, LTR-модель). Все последующие карусели — мгновенно.

In [2]:
from serve_carousel import build_context, serve_one

ctx = build_context()
print()
print('articles loaded:', f'{ctx.art.df.shape[0]:,}')
print('embeddings dim: ', ctx.art.X.shape[1])
print('coview sources: ', f'{len(ctx.coview_neigh_idx):,}')
print('LTR model:      ', type(ctx.model).__name__ if ctx.model else 'None (fallback heuristic)')

[serve] loading articles + embeddings…
      articles: 102,513
[serve] loading LTR model…
      loaded: ltr_lgbm.pkl
[serve] loading coview index…
[coview] reading /Users/nkozheko/T-Ж рекомендации/coview_cache/coview_index.pkl.gz
[coview] aligned sources: 34,746  pairs: 427,001  (0.3s)
[serve] preparing feature cache + explore pools…
      ready in 0.2s

articles loaded: 102,513
embeddings dim:  1024
coview sources:  34,746
LTR model:       LGBMRanker


## 3. Карусель для одной статьи

Берём популярную статью-источник, строим карусель K=12. Колонка scoring_mode показывает, был ли использован LTR (ltr) или fallback (heuristic).

In [3]:
import pandas as pd
pd.set_option('display.max_colwidth', 70)
pd.set_option('display.width', 160)

SOURCE_ID = '56eadad6-91e6-46bc-b889-ad52194d9bf0'  # Что могут сделать судебные приставы

src = ctx.art.df[ctx.art.df['article_id'] == SOURCE_ID].iloc[0]
print('источник:')
print('  title:     ', src.get('article_base__title'))
print('  department:', src.get('article_base__department'))
print('  rubric:    ', src.get('article_base__rubric'))
print()

out = serve_one(ctx, SOURCE_ID)
cols = ['rank', 'candidate_article_id', 'candidate_title', 'candidate_rubric', 'mix', 'similarity', 'score', 'scoring_mode']
out[cols].rename(columns={
    'candidate_title': 'title',
    'candidate_rubric': 'rubric',
    'candidate_article_id': 'cand_id',
}).head(12)

источник:
  title:      Что могут сделать судебные приставы
  department: Право
  rubric:     Другое



,rank,cand_id,title,rubric,mix,similarity,score,scoring_mode
0,1,cad96d3e-80a4-4d1b-9b7d-63c19d806a60,Как я прошла процедуру судебного банкротства: личный опыт от начал...,Материалы из сообщества,similar,0.731400,-0.229352,ltr
1,2,39e247c9-a736-460b-b7d6-577d2a4241f0,Кто такие присяжные,Другое,similar,0.734270,-0.318809,ltr
2,3,258489df-8a1f-4d8c-857d-a766c7ef3d09,С кем останется ребенок после развода,Как жить,similar,0.730474,-0.351639,ltr
3,4,f09164db-3653-4610-8872-ac8a6cc081a6,Президент подписал закон о дополни­тельных выходных и отпусках раб...,,explore,0.428280,-0.124525,ltr
4,5,05f43292-768e-4d6c-96d2-fd7f7f2acb52,Как подать жалобу на судебного пристава,Другое,similar,0.804503,-0.530772,ltr
5,6,d858aa83-5f80-4f74-8955-8cb022659e82,Какие бывают доку­менты-основания при сделках с недви­жимостью,Как жить,similar,0.736499,-0.652893,ltr
6,7,4807eac1-9157-4198-b80f-e33c22aab068,Средняя пенсия по старости по итогам 2025 года составила 27 200,,explore,0.408645,-0.131870,ltr
7,8,8212d9c3-866e-4166-b3b0-d7e15e8a6d80,Подкаст «Схема»: как продать успешный успех,Подкаст «Схема»,similar,0.025000,-0.754882,ltr
8,9,29bee15f-d6b6-4a0c-af8a-01badcec5279,Что такое исполни­тельский сбор,,similar,0.773549,-1.302196,ltr


## 4. Пачка карусели для разных источников

Показывает, что модель работает на разных типах контента (право, финансы, кино, медицина, кулинария…).

In [4]:
SOURCES = [
    ('56eadad6-91e6-46bc-b889-ad52194d9bf0', 'судебные приставы'),
    ('c99f374c-6dff-46b8-8892-1804ea5b3ee4', 'лучшие фильмы 2024'),
    ('4c076836-dede-4dce-81b3-60afefc50c71', 'военная ипотека'),
    ('be91e9bc-e4e6-485c-9ece-770bdd5279e4', 'компенсация за отпуск'),
    ('364718f3-44bb-4d07-89d9-bdef067563c4', 'холестерин и яйца'),
    ('a3bbe8a8-f65b-4af6-bc6d-682662b96b84', 'как выбрать кондиционер'),
]

for sid, label in SOURCES:
    print('=' * 100)
    print(f'SOURCE [{label}]   id={sid}')
    print('=' * 100)
    try:
        out = serve_one(ctx, sid)
    except KeyError as e:
        print(f'  skip: {e}')
        continue
    short = out[['rank', 'candidate_title', 'candidate_rubric', 'mix', 'similarity', 'score', 'scoring_mode']]
    short = short.rename(columns={'candidate_title': 'title', 'candidate_rubric': 'rubric'})
    print(short.to_string(index=False))
    print()

SOURCE [судебные приставы]   id=56eadad6-91e6-46bc-b889-ad52194d9bf0
 rank                                                                                          title                  rubric     mix  similarity     score scoring_mode
    1                   Как я прошла процедуру судебного банкротства: личный опыт от начала до конца Материалы из сообщества similar    0.731400 -0.229352          ltr
    2                                                                            Кто такие присяжные                  Другое similar    0.734270 -0.318809          ltr
    3                                                          С кем останется ребенок после развода                Как жить similar    0.730474 -0.351639          ltr
    4 Президент подписал закон о дополни­тельных выходных и отпусках работни­кам, пострадавшим от ЧС                         explore    0.428280 -0.124525          ltr
    5                                                        Как подать жалобу на судебного

## 5. (опционально) Случайная статья из каталога

Если попросят «покажи на чём-нибудь произвольном» — запускай эту ячейку несколько раз.

In [5]:
import random

views = pd.to_numeric(ctx.art.df.get('article_stats__stats_views', 0), errors='coerce').fillna(0)
df_with_views = ctx.art.df[views > 1000]
sid = random.choice(df_with_views['article_id'].tolist())
src = ctx.art.df[ctx.art.df['article_id'] == sid].iloc[0]
print('source:', src.get('article_base__title'), '  (dept =', src.get('article_base__department'), ')')
print()
out = serve_one(ctx, sid)
out[['rank', 'candidate_title', 'candidate_rubric', 'mix', 'similarity', 'score', 'scoring_mode']].head(12)

source: Как вы работаете по сравнению с другими россиянами?   (dept = Тесты )



,rank,candidate_title,candidate_rubric,mix,similarity,score,scoring_mode
0,1,"Сколько россиян получает больше 100, 500 тысяч или 3 млн в месяц",,similar,0.664407,0.267635,ltr
1,2,"В ИТ 5—6 интервью, в машиностроении надо просто прийти: сколько ищ...",,similar,0.633692,0.151787,ltr
2,3,РАНХиГС: выпуск­ники колледжей и вузов стали чаще работать по спец...,,similar,0.627620,0.113975,ltr
3,4,Насколько вы готовы к серьезным отноше­ниям?,,explore,0.539428,-0.026339,ltr
4,5,За какую прибавку к зарплате люди готовы поменять работу,Другое,similar,0.639761,0.071038,ltr
5,6,"Не получили высшее образование? Расскажите, кем работаете",Дискуссии,similar,0.634103,0.067711,ltr
6,7,Совет путешествен­ника: что привезти из Шанхая,Материалы из сообщества,explore,0.313110,-0.513989,ltr
7,8,Работа и деньги или свободное время: что чаще выбирают люди,Другое,similar,0.649717,0.007911,ltr
8,9,Почему современ­ные выпуск­ники не рабо­тают по специаль­ности,Другое,similar,0.623158,-0.027366,ltr
9,10,Мини-кроссворд № 58: о русских поэтах,,explore,0.441376,-0.815902,ltr


## 6. (опционально) Сравнение LTR vs fallback-эвристика

Выключим LTR и пересоберём ту же карусель эвристикой sim + priors. Видно, что именно дала LTR: перестановки в топе по ltr_score.

In [6]:
saved_model = ctx.model
ctx.model = None  # отключаем LTR
out_heur = serve_one(ctx, SOURCE_ID)
ctx.model = saved_model  # возвращаем обратно

print('--- эвристический скор (sim + priors), scoring_mode = heuristic ---')
print(out_heur[['rank', 'candidate_title', 'mix', 'similarity', 'score', 'scoring_mode']].head(12).to_string(index=False))
print()
print('--- LTR (LightGBM LambdaRank), scoring_mode = ltr ---')
out_ltr = serve_one(ctx, SOURCE_ID)
print(out_ltr[['rank', 'candidate_title', 'mix', 'similarity', 'score', 'scoring_mode']].head(12).to_string(index=False))

--- эвристический скор (sim + priors), scoring_mode = heuristic ---
 rank                                                                  candidate_title     mix  similarity    score scoring_mode
    1                                          Как подать жалобу на судебного пристава similar    0.804503 1.453789    heuristic
    2                                                              Кто такие присяжные similar    0.734270 1.432663    heuristic
    3                                            С кем останется ребенок после развода similar    0.730474 1.344502    heuristic
    4 Как я совмещаю преподавание в вузе, онлайн-репетиторство и уход за пожилой мамой explore    0.477627 0.777233    heuristic
    5                   Какие бывают доку­менты-основания при сделках с недви­жимостью similar    0.736499 1.319625    heuristic
    6     Как я прошла процедуру судебного банкротства: личный опыт от начала до конца similar    0.731400 1.112722    heuristic
    7              Мнение: лю